In [2]:
import torch

In [19]:
def small_tensor_string(
    t, name="", row_limit=50, col_limit=150, min_important_value=1e-5, sci_mode=None, precision=None,
):
    # Unfortunately, pytest usually captures the output and so we can't access the real width here :-(
    # col_limit = col_limit or shutil.get_terminal_size().columns
    shape = "x".join([str(i) for i in t.shape])
    if len(t.shape) > 2:
        t = t.squeeze()
    if len(t.shape) < 2:
        t = t.unsqueeze(0)

    abs = torch.abs(t)
    # Things large enough that we can't round them off to zero and small enough
    # that we need scientific notation to print them or if anything's big enough
    # that we need scientific notation.
    sci_mode = sci_mode if sci_mode is not None else (
        torch.any(torch.logical_and(abs > min_important_value, abs < 1e-3))
        or torch.max(abs) > 1e3
    )
    precision = precision if precision is not None else (2 if sci_mode else 3)

    def fallback():
        with torch._tensor_str.printoptions(
            precision=2 if sci_mode else 3,
            linewidth=col_limit,
            sci_mode=sci_mode,
            threshold=0,
        ):
            return f"{name}[{shape}], {t.dtype}:\n{t}"

    if len(t.shape) > 2 or t.shape[0] > row_limit:
        return fallback()

    def f_entry(d, width=0):
        return f"{d: {width}.{precision}e}" if sci_mode else f"{d: {width}.{precision}f}"

    width = max(len(f_entry(d)) for d in t.flatten().tolist())

    row_width = len(" ".join(f_entry(d, width) for d in t[0].tolist()))

    if row_width > col_limit:
        return fallback()

    rows = []
    for row in t:
        rows.append(" ".join(f_entry(d, width) for d in row.tolist()))

    nl = "\n"
    return f"{name}[{shape}], {t.dtype}:\n{nl.join(rows)}"

In [3]:
def row_str(row):
    return ' '.join(f'{x:3d}' for x in row)

In [4]:
t = torch.arange(256).reshape(16, 16)
for row in t:
    print(row_str(row))

  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15
 16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31
 32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47
 48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63
 64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79
 80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95
 96  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111
112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127
128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159
160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175
176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191
192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207
208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223
224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239
240 241 242 243 244 245 246 247 248 249 

In [5]:
for x in range(64):
  i = x % 16
  j = (x // 16)*4
  has_row = t[i, j:j+4]
  wants_row = t.transpose(0, 1)[i, j:j+4]
  print(f"{x:2d}:   has {row_str(has_row)},   wants {row_str(wants_row)}")

 0:   has   0   1   2   3,   wants   0  16  32  48
 1:   has  16  17  18  19,   wants   1  17  33  49
 2:   has  32  33  34  35,   wants   2  18  34  50
 3:   has  48  49  50  51,   wants   3  19  35  51
 4:   has  64  65  66  67,   wants   4  20  36  52
 5:   has  80  81  82  83,   wants   5  21  37  53
 6:   has  96  97  98  99,   wants   6  22  38  54
 7:   has 112 113 114 115,   wants   7  23  39  55
 8:   has 128 129 130 131,   wants   8  24  40  56
 9:   has 144 145 146 147,   wants   9  25  41  57
10:   has 160 161 162 163,   wants  10  26  42  58
11:   has 176 177 178 179,   wants  11  27  43  59
12:   has 192 193 194 195,   wants  12  28  44  60
13:   has 208 209 210 211,   wants  13  29  45  61
14:   has 224 225 226 227,   wants  14  30  46  62
15:   has 240 241 242 243,   wants  15  31  47  63
16:   has   4   5   6   7,   wants  64  80  96 112
17:   has  20  21  22  23,   wants  65  81  97 113
18:   has  36  37  38  39,   wants  66  82  98 114
19:   has  52  53  54  55,   wa

In [6]:
els_to_thread_and_idx = {}
for x in range(64):
  i = x % 16
  j = (x // 16)*4
  has_row = t[i, j:j+4]
  for v_idx, el in enumerate(has_row.tolist()):
    assert el not in els_to_thread_and_idx
    els_to_thread_and_idx[el] = (x, v_idx)

In [7]:
for x in range(64):
  i = x % 16
  j = (x // 16)*4
  wants_row = t.transpose(0, 1)[i, j:j+4]
  
  wants_idxs = []
  for el in wants_row.tolist():
    wants_idxs.append(els_to_thread_and_idx[el])

  print(f"{x:2d}:  {' '.join(f'{w[0]:2d}:{w[1]}' for w in wants_idxs)}")

 0:   0:0  1:0  2:0  3:0
 1:   0:1  1:1  2:1  3:1
 2:   0:2  1:2  2:2  3:2
 3:   0:3  1:3  2:3  3:3
 4:  16:0 17:0 18:0 19:0
 5:  16:1 17:1 18:1 19:1
 6:  16:2 17:2 18:2 19:2
 7:  16:3 17:3 18:3 19:3
 8:  32:0 33:0 34:0 35:0
 9:  32:1 33:1 34:1 35:1
10:  32:2 33:2 34:2 35:2
11:  32:3 33:3 34:3 35:3
12:  48:0 49:0 50:0 51:0
13:  48:1 49:1 50:1 51:1
14:  48:2 49:2 50:2 51:2
15:  48:3 49:3 50:3 51:3
16:   4:0  5:0  6:0  7:0
17:   4:1  5:1  6:1  7:1
18:   4:2  5:2  6:2  7:2
19:   4:3  5:3  6:3  7:3
20:  20:0 21:0 22:0 23:0
21:  20:1 21:1 22:1 23:1
22:  20:2 21:2 22:2 23:2
23:  20:3 21:3 22:3 23:3
24:  36:0 37:0 38:0 39:0
25:  36:1 37:1 38:1 39:1
26:  36:2 37:2 38:2 39:2
27:  36:3 37:3 38:3 39:3
28:  52:0 53:0 54:0 55:0
29:  52:1 53:1 54:1 55:1
30:  52:2 53:2 54:2 55:2
31:  52:3 53:3 54:3 55:3
32:   8:0  9:0 10:0 11:0
33:   8:1  9:1 10:1 11:1
34:   8:2  9:2 10:2 11:2
35:   8:3  9:3 10:3 11:3
36:  24:0 25:0 26:0 27:0
37:  24:1 25:1 26:1 27:1
38:  24:2 25:2 26:2 27:2
39:  24:3 25:3 26:3 27:3


In [8]:
for x in range(64):
  i = x % 16
  j = (x // 16)*4
  wants_row = t.transpose(0, 1)[i, j:j+4]
  
  wants_el = wants_row.tolist()[0]
  wants_t, wants_v_idx = els_to_thread_and_idx[wants_el]
  row = (x // 16)
  col = (x % 16)
  calc = ((x % 16) // 4) * 16 + (x // 16) * 4
  if wants_v_idx == 0:
    print(f"{x:2d}:  {wants_t:2d} {calc:2d}")

 0:   0  0
 4:  16 16
 8:  32 32
12:  48 48
16:   4  4
20:  20 20
24:  36 36
28:  52 52
32:   8  8
36:  24 24
40:  40 40
44:  56 56
48:  12 12
52:  28 28
56:  44 44
60:  60 60


In [27]:
a = torch.arange(16*16).reshape(16, 16)
# This actually overflows memory, so we need to make the output tensor larger
a_t = torch.zeros(20, 20)

for x in range(64):
    k_idx = x % 16
    n_idx = 4 * (x // 16)
    reg = a[k_idx, n_idx:n_idx+4]
    print(f"{x:2d}: a[{k_idx:2d},{n_idx:2d}:{n_idx+4:<2d}] = {reg}")

    # bad miscompile
    a_t[n_idx, k_idx:k_idx+4] = reg

    print(small_tensor_string(a_t, precision=0, sci_mode=False, col_limit=300))



 0: a[ 0, 0:4 ] = tensor([0, 1, 2, 3])
[20x20], torch.float32:
 0  1  2  3  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  

In [28]:
print(small_tensor_string(a_t, precision=0, sci_mode=False, col_limit=300))

[20x20], torch.float32:
   0   16   32   48   64   80   96  112  128  144  160  176  192  208  224  240  241  242  243    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   4   20   36   52   68   84  100  116  132  148  164  180  196  212  228  244  245  246  247    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   8   24   40   56   72   88  104  120  136  152  168  184  200  216  232  248  249  250  251    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0  